In [9]:
import os
import pandas as pd
import re

case = 'case_study_2_revised'
projdir = "/home/ymerel/fmri/sosym_2026"
resdir = os.path.join(projdir, "results", case)
datadir = os.path.join(projdir, "data", case)

# Function to convert time string to seconds
def time_to_seconds(time_str):
    h, m, s = map(float, re.findall(r'\[?(\d+):(\d+):(\d+\.\d+)\]?', time_str)[0])
    return h * 3600 + m * 60 + s

# Read the log file
with open(os.path.join(datadir, 'elapsed_times.log'), 'r') as file:
    lines = file.readlines()

# Prepare data
data = []
for line in lines:
    line = line.strip()

    # Extract id using regex
    id_match = re.search(r'\[([^\]]+)\]', line)
    if id_match:
        id = id_match.group(1)
    else:
        id = None  # Handle unexpected lines if necessary

    # Extract elapsed time
    elapsed_time_str = re.search(r'Elapsed time \[([^\]]+)\]', line).group(1)
    elapsed = time_to_seconds(elapsed_time_str)

    # Extract cores
    cores_str = re.search(r'\[(\d+)\] cores', line).group(1)
    cores = int(cores_str)

    data.append({"id": id, "elapsed": elapsed, "cores": cores})

# Create DataFrame
df = pd.DataFrame(data)

# Read valid_dataset.csv
valid_df = pd.read_csv(os.path.join(datadir, 'valid_dataset.csv'), sep=';')

# Add 'valid' column
df['valid'] = df['id'].isin(valid_df['id'])

# Write to CSV
df.to_csv(os.path.join(resdir, "elapsed_times.csv"), index=False, sep=';')

valid_count = df['valid'].sum()
print(f"Valid : {valid_count}")

df.head(1001)


Valid : 869


,id,elapsed,cores,valid
0,37dd92ca7ddadf95a5a17a767eb40f0d37e2a28fdb6e0a...,779.005,32,True
1,5c4ebd5a6bb75486ffaa7aecc4fe1b0568938f924a35ab...,712.868,32,True
2,d9da52a92f272e67063ad3f4c0f003565c05c9dfaadc4a...,708.853,32,True
3,76e40d72b9325e876083f6da7512419ba4d73a1aa6def6...,937.087,32,True
4,a0f0593cd25b376403e53368ccfd92668aae08f5ecdb63...,708.844,32,True
...,...,...,...,...
996,ef34468dc2d6605a531fe1d13150c4af1c19b7ee8d6c1a...,480.599,72,True
997,773c731319dabbe3771c540f446cff2368d3b8b6f422f5...,436.558,72,True
998,ab74644f0568416e655ec942b2e3e1946f6c245d38ddd9...,566.695,72,False
999,89d0650eb1e868d2f8ea5e78bd166b4834159d2240775b...,560.690,72,True


In [10]:
# Mean elapsed time for all lines
mean_all = df['elapsed'].mean()

# Mean elapsed time for valid lines
mean_valid = df[df['valid']]['elapsed'].mean()

# Mean elapsed time for non-valid lines
mean_non_valid = df[~df['valid']]['elapsed'].mean()

print(f"Mean elapsed time (all): {mean_all:.2f} seconds")
print(f"Mean elapsed time (valid): {mean_valid:.2f} seconds")
print(f"Mean elapsed time (invalid): {mean_non_valid:.2f} seconds")


Mean elapsed time (all): 627.50 seconds
Mean elapsed time (valid): 613.39 seconds
Mean elapsed time (invalid): 720.40 seconds


In [11]:
# Total elapsed time for all configurations
total_all_seconds = df['elapsed'].sum()

# Total elapsed time for valid configurations
total_valid_seconds = df[df['valid']]['elapsed'].sum()

# Total elapsed time for non-valid configurations
total_non_valid_seconds = df[~df['valid']]['elapsed'].sum()

# Function to convert seconds to HH:mm:ss
def format_time(seconds):
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    seconds = int(seconds % 60)
    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"

# Format the total times
formatted_all = format_time(total_all_seconds)
formatted_valid = format_time(total_valid_seconds)
formatted_non_valid = format_time(total_non_valid_seconds)

print(f"Total execution time (all configs): {formatted_all}")
print(f"Total execution time (valid configs): {formatted_valid}")
print(f"Total execution time (non-valid configs): {formatted_non_valid} ({(total_non_valid_seconds / total_all_seconds) * 100} %)")

Total execution time (all configs): 174:28:49
Total execution time (valid configs): 148:03:57
Total execution time (non-valid configs): 26:24:52 (15.139011403610898 %)
